In [ ]:
!pip install -q transformers bitsandbytes peft accelerate datasets

import warnings
warnings.filterwarnings("ignore")

categories = [
    "ORDER", "SHIPPING", "CANCEL", "INVOICE",
    "PAYMENT", "REFUND", "FEEDBACK", "CONTACT",
    "ACCOUNT", "DELIVERY", "SUBSCRIPTION"
]

intents = {
    "ORDER": ["cancel_order", "change_order", "place_order", "track_order"],
    "SHIPPING": ["change_shipping_address", "set_up_shipping_address"],
    "CANCEL": ["check_cancellation_fee"],
    "INVOICE": ["check_invoice", "get_invoice"],
    "PAYMENT": ["check_payment_methods", "payment_issue"],
    "REFUND": ["check_refund_policy", "get_refund", "track_refund"],
    "FEEDBACK": ["complaint", "review"],
    "CONTACT": ["contact_customer_service", "contact_human_agent"],
    "ACCOUNT": ["create_account", "delete_account", "edit_account", "recover_password", "registration_problems", "switch_account"],
    "DELIVERY": ["delivery_options", "delivery_period"],
    "SUBSCRIPTION": ["newsletter_subscription"]
}

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

def load_and_split_dataset(dataset_path, dataset_size=None, min_intent_count=5, upsample=True, rare_intent_threshold=2):
    df = pd.read_csv(dataset_path)

    # Filter out very rare intents based on rare_intent_threshold
    intent_counts = df['intent'].value_counts()
    df = df[df['intent'].isin(intent_counts[intent_counts >= rare_intent_threshold].index)]

    # If a smaller dataset size is specified, sample it while keeping stratification
    if dataset_size is not None and dataset_size < len(df):
        df, _ = train_test_split(
            df,
            train_size=dataset_size,
            stratify=df['category'] + '_' + df['intent'],
            random_state=42
        )

    # Split the dataset into training (80%) and temporary (20%) sets
    df_train, df_temp = train_test_split(
        df,
        test_size=0.2,
        stratify=df['category'] + '_' + df['intent'],
        random_state=42
    )

    # Split the temporary set into validation (10%) and test (10%) sets
    df_val, df_test = train_test_split(
        df_temp,
        test_size=0.5,
        stratify=df_temp['category'] + '_' + df_temp['intent'],
        random_state=42
    )

    # Balance intent distribution within each split
    def balance_intents(df_split, min_intent_count=5, upsample=False, drop=True):
        balanced = []
        for category in df_split['category'].unique():
            df_cat = df_split[df_split['category'] == category]
            intent_counts = df_cat['intent'].value_counts()

            for intent, count in intent_counts.items():
                df_group = df_cat[df_cat['intent'] == intent]
                
                if count < min_intent_count:
                    if upsample:
                        df_group = resample(
                            df_group,
                            replace=True,
                            n_samples=min_intent_count,
                            random_state=42
                        )
                        balanced.append(df_group)
                    elif not drop:
                        # Keep small intents without upsampling
                        balanced.append(df_group)
                else:
                    balanced.append(df_group)

        if not balanced:
            return pd.DataFrame(columns=df_split.columns)

        return pd.concat(balanced).sample(frac=1, random_state=42).reset_index(drop=True)

    df_train = balance_intents(df_train, min_intent_count=min_intent_count, upsample=upsample)
    df_val   = balance_intents(df_val, min_intent_count=min_intent_count, upsample=False, drop=False)
    df_test  = balance_intents(df_test, min_intent_count=min_intent_count, upsample=False, drop=False)

    return df_train, df_val, df_test

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

def model_loader(model_id):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        # Configure 4-bit quantization for more efficient GPU memory usage
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        
        # Load the model with quantization
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map={"": 0},
            quantization_config=bnb_config,
            torch_dtype=torch.float16
        )
        
        device = torch.device("cuda")

    else:
        # Load the model without quantization
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
        )
        
        device = torch.device("cpu")

    # Create a text generation pipeline
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

    return model, tokenizer, generator, device

In [ ]:
import re

# Format the prompt for category classification
def format_category_prompt(instruction, categories):
    system_prompt = {
        "role": "system",
        "content": (
            "You are an AI assistant that classifies a user support query into one of the predefined categories.\n\n"
            "### Available Categories:\n"
            f"{chr(10).join(f'- {c}' for c in categories)}\n\n"
            "### Instructions:\n"
            "1. Carefully read the user’s query.\n"
            "2. Choose the **one category** from the list that best matches the query.\n"
            "3. Return only the category name — exactly as shown in the list.\n\n"
            "### Rules:\n"
            "- Do not invent new categories.\n"
            "- Do not include explanations, punctuation, or any extra words.\n"
            "- Output just the category name."
        )
    }

    user_prompt = {
        "role": "user",
        "content": instruction
    }

    assistant_prompt = {
        "role": "assistant",
        "content": "Labeled Category:"
    }
     
    return [system_prompt, user_prompt, assistant_prompt]

# Format the prompt for intent classification
def format_intent_prompt(instruction, intents_for_category):
    system_prompt = {
        "role": "system",
        "content": (
            "You are an AI assistant that selects the correct intent for a user query based on a given category.\n\n"
            "### Available Intents:\n"
            f"{chr(10).join(f'- {intent}' for intent in intents_for_category)}\n\n"
            "### Instructions:\n"
            "1. Read the user query carefully.\n"
            "2. Pick the **single intent** from the list that best matches the query.\n"
            "3. Return only the intent name — exactly as written above.\n\n"
            "### Rules:\n"
            "- Do not make up new intents.\n"
            "- Do not add any extra text, punctuation, or explanations.\n"
            "- Only return the intent name."
        )
    }

    user_prompt = {
        "role": "user",
        "content": instruction
    }

    assistant_prompt = {
        "role": "assistant",
        "content": "Labeled Intent:"
    }

    return [system_prompt, user_prompt, assistant_prompt]

# Predict the category of a user query
def predict_category(instruction, categories, generator):
    messages = format_category_prompt(instruction, categories)
    prompt = "\n".join(f"<|{m['role']}|>\n{m['content']}" for m in messages)
    output = generator(prompt, max_new_tokens=10)[0]["generated_text"]

    # Extract category name from the generated output
    match = re.search(r"Labeled Category:\s*(.*)", output, re.IGNORECASE)
    if match:
        label_section = match.group(1).strip().upper()
        
        for cat in categories:
            if cat in label_section:
                return cat
            
    return "NOT FOUND"

# Predict the intent of a query based on the predicted category
def predict_intent(predicted_category, instruction, intents, generator):
    if predicted_category not in intents:
        all_intents = [intent for sublist in intents.values() for intent in sublist]
        intents_for_cat = all_intents
    else:
        intents_for_cat = intents[predicted_category]
        
    # If there's only one possible intent, return it directly    
    if len(intents_for_cat) == 1:
        return intents_for_cat[0]

    messages = format_intent_prompt(instruction, intents_for_cat)
    prompt = "\n".join(f"<|{m['role']}|>\n{m['content']}" for m in messages)
    output = generator(prompt, max_new_tokens=15)[0]["generated_text"]

    # Extract intent from the generated output
    match = re.search(r"Labeled Intent:\s*(.*)", output, re.IGNORECASE)
    if match:
        label_section = match.group(1).strip().lower()
        
        for intent in intents_for_cat:
            if intent in label_section:
                return intent

    return "NOT FOUND"

# Evaluate category and intent predictions on test data
def general_inference_mistral_7b_v0_2(df_test, generator, categories, intents):
    correct_category = 0
    correct_intent = 0
    total = len(df_test)
    predictions = []

    for _, row in df_test.iterrows():
        instruction = row["instruction"]
        actual_category = row["category"].upper()
        actual_intent = row["intent"].lower()

        predicted_category = predict_category(instruction, categories, generator)
        is_correct_category = predicted_category == actual_category
        correct_category += is_correct_category

        predicted_intent = predict_intent(predicted_category, instruction, intents, generator)
        is_correct_intent = predicted_intent == actual_intent
        correct_intent += is_correct_intent

        predictions.append({
            "instruction": instruction,
            "actual_category": actual_category,
            "predicted_category": predicted_category,
            "actual_intent": actual_intent,
            "predicted_intent": predicted_intent,
        })

    accuracy_category = correct_category / total * 100
    accuracy_intent = correct_intent / total * 100

    return accuracy_category, accuracy_intent, predictions

In [ ]:
import re
from datasets import Dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Format the training data for category classification
def format_chat_category(example, tokenizer):
    example["text"] = (
        f"Instruction: {example['instruction']}\n"
        f"Labeled Category: {example['category']}{tokenizer.eos_token}"
    )
    return example

# Format the training data for intent classification (includes list of possible intents for that category)
def format_chat_intent(example, tokenizer, intents):
    category = example["category"]
    category_intents = intents.get(category.upper(), [])
    intents_str = ", ".join(category_intents)
    example["text"] = (
        f"Instruction: {example['instruction']}\n"
        f"Category: {category}\n"
        f"Possible Intents: {intents_str}\n"
        f"Labeled Intent: {example['intent']}{tokenizer.eos_token}"
    )
    return example

# Tokenize the formatted text for model input
def tokenize_function(example, tokenizer, max_length=512):
    tokenized = tokenizer(example["text"], padding="max_length", truncation=True, max_length=max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Fine-tune for category classification
def finetune_category_mistral_7b_v0_2(df_train, model, tokenizer, device, lora_params: dict, training_params: dict):
    dataset = Dataset.from_pandas(df_train[["instruction", "category"]])
    dataset = dataset.map(lambda x: format_chat_category(x, tokenizer))
    dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=int(lora_params["r"]),
        lora_alpha=int(lora_params["lora_alpha"]),
        target_modules=["q_proj", "v_proj"],
        lora_dropout=float(lora_params["lora_dropout"]),
        bias="none", 
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    training_args = TrainingArguments(
        output_dir="./tinyllama-lora-category-classifier",
        per_device_train_batch_size=int(training_params["batch_size"]),
        gradient_accumulation_steps=int(training_params["gradient_accumulation_steps"]),
        learning_rate=float(training_params["learning_rate"]),
        lr_scheduler_type="cosine",
        warmup_ratio=float(training_params["warmup_ratio"]),
        num_train_epochs=int(training_params["train_epochs"]),
        logging_steps=int(training_params["logging_steps"]),
        save_strategy="epoch",
        report_to="none",
        fp16=True,
        label_names=["labels"]
    )
    
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    trainer = Trainer(
        model=model, args=training_args, train_dataset=dataset,
        tokenizer=tokenizer, data_collator=data_collator
    )
    
    trainer.train()
    return model

# Fine-tune for intent classification
def finetune_intent_mistral_7b_v0_2(df_train, model, tokenizer, device, intents, lora_params: dict, training_params: dict):
    dataset = Dataset.from_pandas(df_train[["instruction", "category", "intent"]])
    dataset = dataset.map(lambda x: format_chat_intent(x, tokenizer, intents))
    dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    
    model = prepare_model_for_kbit_training(model)
    
    lora_config = LoraConfig(
        r=int(lora_params["r"]),
        lora_alpha=int(lora_params["lora_alpha"]),
        target_modules=["q_proj", "v_proj"],
        lora_dropout=float(lora_params["lora_dropout"]),
        bias="none", 
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    training_args = TrainingArguments(
        output_dir="./tinyllama-lora-category-classifier",
        per_device_train_batch_size=int(training_params["batch_size"]),
        gradient_accumulation_steps=int(training_params["gradient_accumulation_steps"]),
        learning_rate=float(training_params["learning_rate"]),
        lr_scheduler_type="cosine",
        warmup_ratio=float(training_params["warmup_ratio"]),
        num_train_epochs=int(training_params["train_epochs"]),
        logging_steps=int(training_params["logging_steps"]),
        save_strategy="epoch",
        report_to="none",
        fp16=True,
        label_names=["labels"]
    )
    
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    trainer = Trainer(
        model=model, args=training_args, train_dataset=dataset,
        tokenizer=tokenizer, data_collator=data_collator
    )
    
    trainer.train()
    return model

# Generate category prediction from instruction
def predict_output_category(text, model, tokenizer, device):
    prompt = (
        f"Instruction: {text}\n"
        f"Labeled Category:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate intent prediction using predicted category and instruction
def predict_output_intent(text, predicted_category, model, tokenizer, device, intents):
    possible_intents = intents.get(predicted_category.upper(), [])
    if not possible_intents:
        possible_intents = [intent for sub in intents.values() for intent in sub]
    
    intents_str = ", ".join(possible_intents)
    prompt = (
        f"Instruction: {text}\n"
        f"Category: {predicted_category}\n"
        f"Possible Intents: {intents_str}\n"
        f"Labeled Intent:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract the predicted category string from the model output
def extract_category(text, categories):
    match = re.search(r"Labeled Category:\s*(.*)", text, re.IGNORECASE)

    if match:
        label_section = match.group(1).strip().upper()
    
        for cat in categories:
            if cat in label_section:
                return cat
                
    return "NOT FOUND"

# Extract the predicted intent string from the model output
def extract_intent(text, intents):
    match = re.search(r"Labeled Intent:\s*(.*)", text, re.IGNORECASE)
    
    if match:
        label_section = match.group(1).strip().lower()
        
        for intent in intents:
            if intent in label_section:
                return intent
                
    return "NOT FOUND"

# Evaluate both models on a test dataset
def evaluate_model_mistral_7b_v0_2(df_test, model_category, model_intent, tokenizer, device, categories, intents):
    predictions = []
    correct_category = 0
    correct_intent = 0
    total = len(df_test)
    
    all_intents_flat = [intent.lower() for sublist in intents.values() for intent in sublist]
    
    for _, row in df_test.iterrows():
        instruction = row["instruction"]
        actual_category = row["category"].strip().upper()
        actual_intent = row["intent"].strip().lower()
        
        output_category = predict_output_category(instruction, model_category, tokenizer, device)
        pred_category = extract_category(output_category, categories).strip().upper()

        category_intents = intents.get(pred_category, [])

        if len(category_intents) == 1:
            pred_intent = category_intents[0].strip().lower()
        else:
            output_intent = predict_output_intent(instruction, pred_category, model_intent, tokenizer, device, intents)
            pred_intent = extract_intent(output_intent, all_intents_flat).strip().lower()
        
        if pred_category == actual_category:
            correct_category += 1
        if pred_intent == actual_intent:
            correct_intent += 1
            
        predictions.append({
            "instruction": instruction,
            "actual_category": actual_category,
            "predicted_category": pred_category,
            "actual_intent": actual_intent,
            "predicted_intent": pred_intent,
        })
        
    accuracy_category = (correct_category / total) * 100
    accuracy_intent = (correct_intent / total) * 100
    
    return accuracy_category, accuracy_intent, predictions

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
from huggingface_hub import login

# Log in to Hugging Face Hub using an access token
hf_token = "your_hf_key"
login(token=hf_token)

# LoRA parameters for fine-tuning
lora_parameters = {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05
}

# Training parameters for fine-tuning
training_parameters = {
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1e-4,
    "warmup_ratio": 0.03,
    "train_epochs": 3,
    "logging_steps": 100
}

# Dataset path and size configuration
dataset_path = 'path_to_dataset'
dataset_size = 300 # No of samples to apply. Use full dataset (27k rows) by setting to None

# Load and split the dataset
df_train, df_val, df_test = load_and_split_dataset(dataset_path, dataset_size)

# Load the initial model, tokenizer, and text generation pipeline
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
model_initial, tokenizer_initial, generator, device = model_loader(model_id)

# ============================================================
# CASE 1: Performing inference using general prompting (no fine-tuning)
# ============================================================
accuracy_category, accuracy_intent, predictions_inference = general_inference_mistral_7b_v0_2(df_test, generator, categories, intents)

print(f"Accuracy without Fine-tuning [Mistral 7B v0.2] on categories: {accuracy_category:.2f}%")
print(f"Accuracy without Fine-tuning [Mistral 7B v0.2] on intents: {accuracy_intent:.2f}%")

# ============================================================
# CASE 2: Performing fine-tuning with LoRA
# ============================================================
model_category_llama_3_2_3b = finetune_category_mistral_7b_v0_2(df_train, model_initial, tokenizer_initial, device, lora_params=lora_parameters, training_params=training_parameters)
model_intent_llama_3_2_3b = finetune_intent_mistral_7b_v0_2(df_train, model_initial, tokenizer_initial, device, intents, lora_params=lora_parameters, training_params=training_parameters)

accuracy_category, accuracy_intent, predictions_finetune = evaluate_model_mistral_7b_v0_2(df_test, model_category_llama_3_2_3b, model_intent_llama_3_2_3b, tokenizer_initial, device, categories, intents)

print(f"Accuracy with LoRa Fine-tuning [Mistral 7B v0.2] on categories: {accuracy_category:.2f}%")
print(f"Accuracy with LoRa Fine-tuning [Mistral 7B v0.2] on intents: {accuracy_intent:.2f}%")